# LoRA from Scratch: Why LLM Fine-Tuning Got 1000x Cheaper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/llm/lora_qlora_from_scratch.ipynb)

Companion notebook to the [blog post](https://sesen.ai/blog/lora-qlora-parameter-efficient-fine-tuning). LoRA and QLoRA in pure PyTorch, all four experiments:

1. LoRA layer in 20 lines, then quick-win Task A → Task B fine-tune
2. Rank ablation: r=1, 2, 4, 8, 16, 32 vs full FT
3. Where to inject LoRA: FFN vs head vs everything
4. QLoRA: 4-bit base quantisation + LoRA adapter

All experiments run on CPU in seconds. No HuggingFace, no model downloads.

In [ ]:
import math
import copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

torch.manual_seed(0)
np.random.seed(0)

## 1. Base "pretrained" model

A 4-block FFN-style network — stand-in for an LLM at toy scale. The LoRA mechanic transfers identically to real transformer weight matrices.

In [ ]:
class Block(nn.Module):
    def __init__(self, d_model, d_hidden):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_hidden)
        self.fc2 = nn.Linear(d_hidden, d_model)
        self.norm = nn.LayerNorm(d_model)
    def forward(self, x):
        return self.norm(x + self.fc2(F.relu(self.fc1(x))))

class BaseModel(nn.Module):
    def __init__(self, d_in=10, d_model=64, d_hidden=128, n_blocks=4, n_classes=4):
        super().__init__()
        self.embed = nn.Linear(d_in, d_model)
        self.blocks = nn.ModuleList([Block(d_model, d_hidden) for _ in range(n_blocks)])
        self.head = nn.Linear(d_model, n_classes)
    def forward(self, x):
        h = self.embed(x)
        for block in self.blocks:
            h = block(h)
        return self.head(h)

## 2. LoRA layer (Hu et al. 2021)

Wraps a frozen `nn.Linear` with two trainable low-rank matrices $A \in \mathbb{R}^{r \times k}$ and $B \in \mathbb{R}^{d \times r}$. Forward: $h = W_0 x + (\alpha/r) B A x$.

Initialisation: A from Kaiming uniform, B from zero, so the LoRA update is initially a no-op.

In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, r: int = 8, alpha: float = 16.0):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad = False
        self.r, self.scaling = r, alpha / r
        in_f, out_f = base.in_features, base.out_features
        self.A = nn.Parameter(torch.empty(r, in_f))
        self.B = nn.Parameter(torch.zeros(out_f, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
    def forward(self, x):
        return self.base(x) + self.scaling * (x @ self.A.T) @ self.B.T

In [ ]:
def inject_lora(model, r=8, alpha=16.0, target='all'):
    """Replace selected nn.Linear children with LoRALinear (two-pass to avoid recursion)."""
    replacements = []
    for parent_name, parent in list(model.named_modules()):
        for sub_name, sub in list(parent.named_children()):
            if not isinstance(sub, nn.Linear):
                continue
            full = f'{parent_name}.{sub_name}' if parent_name else sub_name
            if target == 'ffn' and sub_name not in {'fc1', 'fc2'}: continue
            if target == 'fc1_only' and sub_name != 'fc1': continue
            if target == 'head' and sub_name != 'head': continue
            if target == 'no_head' and 'head' in full: continue
            replacements.append((parent, sub_name, sub))
    for parent, sub_name, sub in replacements:
        setattr(parent, sub_name, LoRALinear(sub, r=r, alpha=alpha))
    for p in model.parameters():
        p.requires_grad = False
    for name, p in model.named_parameters():
        if name.split('.')[-1] in {'A', 'B'}:
            p.requires_grad = True
    return model

def count_trainable(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)
def freeze(m): 
    for p in m.parameters(): p.requires_grad = False
def unfreeze(m):
    for p in m.parameters(): p.requires_grad = True

def train(model, X_tr, y_tr, X_te, y_te, n_epochs=80, lr=0.005):
    Xt, yt = torch.from_numpy(X_tr).float(), torch.from_numpy(y_tr).long()
    Xte, yte = torch.from_numpy(X_te).float(), torch.from_numpy(y_te).long()
    params = [p for p in model.parameters() if p.requires_grad]
    if not params:
        with torch.no_grad():
            return float((model(Xte).argmax(-1).numpy() == y_te).mean())
    opt = torch.optim.Adam(params, lr=lr)
    for _ in range(n_epochs):
        opt.zero_grad()
        F.cross_entropy(model(Xt), yt).backward()
        opt.step()
    with torch.no_grad():
        return float((model(Xte).argmax(-1).numpy() == y_te).mean())

## 3. Quick-win: pretrain on Task A, fine-tune on Task B

Task A: 4-class classification in 10D. Task B: same generative process but rotated, mimicking a related downstream task.

In [ ]:
# Task A
X_a, y_a = make_classification(n_samples=4000, n_features=10, n_informative=6,
                                n_redundant=2, n_classes=4, n_clusters_per_class=1,
                                class_sep=1.4, random_state=0)
X_a, y_a = X_a.astype(np.float32), y_a.astype(np.int64)
X_a_tr, X_a_te, y_a_tr, y_a_te = train_test_split(X_a, y_a, test_size=0.25, random_state=0)

# Pretrain base
base = BaseModel()
pre_acc = train(base, X_a_tr, y_a_tr, X_a_te, y_a_te, n_epochs=60)
print(f'Task A test accuracy after pretraining: {pre_acc:.4f}')
freeze(base)

# Task B (rotated version of similar problem)
X_b, y_b = make_classification(n_samples=2000, n_features=10, n_informative=6,
                                n_redundant=2, n_classes=4, n_clusters_per_class=1,
                                class_sep=1.4, random_state=11)
Q, _ = np.linalg.qr(np.random.default_rng(0).standard_normal((10, 10)))
X_b = (X_b @ Q.T).astype(np.float32)
y_b = y_b.astype(np.int64)
X_b_tr, X_b_te, y_b_tr, y_b_te = train_test_split(X_b, y_b, test_size=0.25, random_state=0)

# Compare three strategies
m_frozen = copy.deepcopy(base); freeze(m_frozen)
acc_frozen = train(m_frozen, X_b_tr, y_b_tr, X_b_te, y_b_te, n_epochs=0)

m_full = copy.deepcopy(base); unfreeze(m_full)
acc_full = train(m_full, X_b_tr, y_b_tr, X_b_te, y_b_te, n_epochs=80)

m_lora = copy.deepcopy(base)
inject_lora(m_lora, r=8, alpha=16, target='ffn')
acc_lora = train(m_lora, X_b_tr, y_b_tr, X_b_te, y_b_te, n_epochs=80)

print(f'\nTask B test accuracy:')
print(f'  Frozen (no FT):  {acc_frozen:.4f}  (0 trainable)')
print(f'  Full FT:         {acc_full:.4f}  ({count_trainable(m_full):,} trainable)')
print(f'  LoRA r=8 on FFN: {acc_lora:.4f}  ({count_trainable(m_lora):,} trainable, '
      f'{count_trainable(m_full)/max(count_trainable(m_lora),1):.0f}x fewer)')

## 4. Rank ablation

Sweep r ∈ {1, 2, 4, 8, 16, 32} and watch the accuracy plateau.

In [ ]:
for r in [1, 2, 4, 8, 16, 32]:
    m = copy.deepcopy(base)
    inject_lora(m, r=r, alpha=2*r, target='ffn')
    acc = train(m, X_b_tr, y_b_tr, X_b_te, y_b_te, n_epochs=80)
    print(f'  r={r:>2}: trainable={count_trainable(m):>7,}, acc={acc:.4f}')

## 5. Where to inject LoRA

FFN vs head vs everything except head. The lift is overwhelmingly in the FFN, not the output layer.

In [ ]:
for name, t in [('FFN fc1 only', 'fc1_only'),
                 ('All FFN linears', 'ffn'),
                 ('Head only', 'head'),
                 ('Everything except head', 'no_head')]:
    m = copy.deepcopy(base)
    inject_lora(m, r=8, alpha=16, target=t)
    acc = train(m, X_b_tr, y_b_tr, X_b_te, y_b_te, n_epochs=80)
    print(f'  {name:30s}  trainable={count_trainable(m):>7,}  acc={acc:.4f}')

## 6. QLoRA: 4-bit quantisation of the frozen base

Quantise base weights to 4 bits. LoRA adapters stay in FP32. Real QLoRA uses NormalFloat4; this uniform-quantile version captures the same accuracy/memory trade-off.

In [ ]:
def quantise_4bit(W, n_levels=16):
    absmax = W.abs().max() + 1e-9
    scale = absmax / ((n_levels - 1) / 2)
    q = torch.round(W / scale).clamp(-(n_levels // 2), n_levels // 2 - 1)
    return q * scale

def apply_4bit_quantisation(model):
    with torch.no_grad():
        for module in model.modules():
            if isinstance(module, nn.Linear) and not module.weight.requires_grad:
                module.weight.copy_(quantise_4bit(module.weight))

m_qlora = copy.deepcopy(base)
inject_lora(m_qlora, r=8, alpha=16, target='ffn')
apply_4bit_quantisation(m_qlora)
acc_qlora = train(m_qlora, X_b_tr, y_b_tr, X_b_te, y_b_te, n_epochs=80)

print(f'Full FT (FP32):          acc={acc_full:.4f}')
print(f'LoRA r=8 (FP32 base):    acc={acc_lora:.4f}')
print(f'QLoRA r=8 (4-bit base):  acc={acc_qlora:.4f}')

base_params = sum(p.numel() for p in base.parameters())
fp32_size_mb = base_params * 4 / (1024 ** 2)
int4_size_mb = base_params * 0.5 / (1024 ** 2)
print(f'\nBase storage:')
print(f'  FP32: {fp32_size_mb:.3f} MB')
print(f'  INT4: {int4_size_mb:.3f} MB  ({fp32_size_mb / int4_size_mb:.0f}x compression)')

## Exercises

1. **LoRA merging**: After fine-tuning, compute `W = base.weight + (alpha/r) * B @ A` and verify the merged-weight model gives bit-exact identical outputs to the LoRA-injected model. This is what makes LoRA zero-overhead at inference.

2. **DoRA (Decomposed LoRA)**: The 2024 follow-up decomposes pretrained weights into magnitude × direction and lets LoRA adapt the direction only. Implement this and compare to vanilla LoRA on the same task. (Liu et al., DoRA paper.)

3. **NormalFloat4**: Replace `quantise_4bit` with a 16-bin codebook calibrated to a standard normal distribution (the codebook values are $\Phi^{-1}((i + 0.5)/16)$ for i ∈ 0..15). Measure the accuracy gap closure compared to the uniform 4-bit version.

4. **LoRA on attention**: Build a tiny multi-head attention module (Q, K, V, O projections) and LoRA-inject only Q and V (Hu et al.'s standard recipe). Compare to LoRA on FFN. Which is more parameter-efficient on the same task?

5. **Adapter swapping**: Train two LoRA adapters on two different downstream tasks. Verify that you can swap adapters at inference time without reloading the frozen base, just by replacing the A and B parameters.